# Unitree A1 — MJX/Brax GPU Training

**Before running:** Go to `Runtime → Change runtime type → GPU (T4 or A100)`

Then run all cells in order.

In [ ]:
# ── 1. Check Accelerator ────────────────────────────────────────────────────────
import subprocess
try:
    result = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True)
    print('GPU:', result.stdout.strip())
except FileNotFoundError:
    print('GPU: nvidia-smi not found. (Running on TPU or CPU runtime)')

import jax
print('JAX version:', jax.__version__)
print('JAX devices:', jax.devices())


In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
!pip install -q mujoco mujoco-mjx brax mediapy wandb
print('Done')

In [ ]:
# ── 3. Mount Google Drive (persists checkpoints across Colab sessions) ────────
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/unitree_a1_mjx'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Checkpoints will be saved to:', DRIVE_DIR)

In [ ]:
# ── 4. Upload the Unitree A1 XML assets ──────────────────────────────────────
# Upload ALL files from external/mujoco_menagerie/unitree_a1/
# (scene.xml + a1.xml + assets/ meshes if any)
from google.colab import files
import os

PROJECT_DIR = '/content/graduation_project'
XML_DIR = f'{PROJECT_DIR}/external/mujoco_menagerie/unitree_a1'
os.makedirs(XML_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/lab_mjx', exist_ok=True)

print(f'Select all files from your local external/mujoco_menagerie/unitree_a1/ folder:')
uploaded = files.upload()
for fname, data in uploaded.items():
    dest = os.path.join(XML_DIR, os.path.basename(fname))
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  Saved: {dest}')
print('\nFiles uploaded:', os.listdir(XML_DIR))

In [ ]:
# ── 5. Upload or Sync env.py ──────────────────────────────────────────────────
import os
import shutil
from google.colab import files

PROJECT_DIR = '/content/graduation_project'
local_env_dir = f'{PROJECT_DIR}/lab_mjx'
local_env_path = f'{local_env_dir}/env.py'
drive_env_path = '/content/drive/MyDrive/unitree_a1_mjx/env.py'

# Set to True to force a fresh upload of env.py from your local machine
# (e.g. after making local edits that need to override the Google Drive version)
FORCE_UPLOAD = False

os.makedirs(local_env_dir, exist_ok=True)

if os.path.exists(drive_env_path) and not FORCE_UPLOAD:
    shutil.copy(drive_env_path, local_env_path)
    print(f"Loaded env.py from Google Drive: {drive_env_path}")
else:
    if FORCE_UPLOAD:
        print("FORCE_UPLOAD is True. Please upload your updated local env.py:")
    else:
        print("env.py not found on Google Drive. Please upload env.py from your local lab_mjx/ folder:")
    
    uploaded = files.upload()
    for fname, data in uploaded.items():
        if fname.endswith('env.py'):
            with open(local_env_path, 'wb') as f:
                f.write(data)
            print(f"Saved: {local_env_path}")
            # Save/Update the copy on Drive to persist for future sessions
            if os.path.exists('/content/drive/MyDrive/unitree_a1_mjx'):
                shutil.copy(local_env_path, drive_env_path)
                print(f"Saved/Updated copy to Google Drive: {drive_env_path}")
            break


In [ ]:
# ── 6. WandB login (Robust for Interactive and Background Runs) ────────────────
import os
import wandb

# Try to fetch WandB API key from Kaggle Secrets if running on Kaggle
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret('wandb_key')
    if wandb_key:
        os.environ['WANDB_API_KEY'] = wandb_key
        print('Successfully configured WandB API Key from Kaggle Secrets.')
except Exception:
    # Not on Kaggle, or secret not set
    pass

try:
    # Attempt to login
    wandb.login(timeout=10)
except Exception as e:
    print(f'WandB login could not prompt for key (background run or no internet): {e}')
    print('Setting WANDB_MODE to offline to prevent crash. Training will run normally.')
    os.environ['WANDB_MODE'] = 'offline'


In [ ]:
# ── 7. GPU performance flags + training ───────────────────────────────────────
import os, time, pickle, functools, sys
os.environ['XLA_FLAGS'] = os.environ.get('XLA_FLAGS','') + ' --xla_gpu_triton_gemm_any=True'
os.environ['JAX_DEFAULT_MATMUL_PRECISION'] = 'highest'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

import jax
import jax.numpy as jp
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks

sys.path.insert(0, f'{PROJECT_DIR}/lab_mjx')
from env import UnitreeA1MJXEnv

XML_PATH      = f'{PROJECT_DIR}/external/mujoco_menagerie/unitree_a1/scene.xml'
NUM_ENVS      = 4096   # T4: 4096 | A100: 8192 | RTX3060: 2048
NUM_TIMESTEPS = 100_000_000  # 100M is standard for locomotion convergence
EPISODE_LEN   = 500
POLICY_H      = (256, 256)
VALUE_H       = (256, 256)
VERSION       = '1.0'
NUM_EVALS     = 100    # Evaluate every 1M steps for quick feedback

env = UnitreeA1MJXEnv(XML_PATH)
print(f'Devices: {jax.devices()}  obs={env.observation_size}  act={env.action_size}')

net_factory = functools.partial(ppo_networks.make_ppo_networks,
    policy_hidden_layer_sizes=POLICY_H, value_hidden_layer_sizes=VALUE_H)

t0 = time.perf_counter()

def progress_fn(step, metrics):
    el = time.perf_counter()-t0
    rew = float(metrics.get('eval/episode_reward', float('nan')))
    print(f'[{el/60:.1f}m | {step:>11,} | {step/max(el,1):>7,.0f} sps] reward={rew:.3f}')
    try: wandb.log({k: float(v) for k,v in metrics.items()}, step=step)
    except: pass

run = wandb.init(project='unitree_a1_rl', name=f'a1_mjx_v{VERSION}_colab',
                 config=dict(num_envs=NUM_ENVS, num_timesteps=NUM_TIMESTEPS, num_evals=NUM_EVALS))

print('Starting training (first compile ~2-4 min)...')
make_policy, params, metrics = ppo.train(
    environment=env, num_timesteps=NUM_TIMESTEPS, num_envs=NUM_ENVS,
    episode_length=EPISODE_LEN, num_evals=NUM_EVALS, reward_scaling=1.0,
    unroll_length=20, batch_size=512, num_minibatches=8, num_updates_per_batch=4,
    discounting=0.99, learning_rate=3e-4, entropy_cost=0.01, gae_lambda=0.95,
    max_grad_norm=0.5, clipping_epsilon=0.2, normalize_observations=True,
    normalize_observations_std_eps=1e-5,  # Prevents division by zero for constant features
    network_factory=net_factory, progress_fn=progress_fn, seed=42)

run.finish()
print('\nTraining complete!')

In [ ]:
# ── 8. Save Model Parameters and Configuration ───────────────────────────────
import pickle
param_path = f'{DRIVE_DIR}/a1_mjx_v{VERSION}_params.pkl'
config_path = f'{DRIVE_DIR}/a1_mjx_v{VERSION}_config.pkl'

with open(param_path, 'wb') as f:
    pickle.dump(params, f)
with open(config_path, 'wb') as f:
    pickle.dump({
        'policy_hidden': POLICY_H,
        'value_hidden': VALUE_H,
        'obs_size': env.observation_size,
        'act_size': env.action_size
    }, f)

print(f'Saved parameters to: {param_path}')
print(f'Saved config to: {config_path}')
if IS_COLAB:
    print('Saved to Google Drive!')
else:
    print('Saved to Kaggle Output folder! You can download both .pkl files directly from the Kaggle right-hand panel.')


In [ ]:
# ── 9. Inline video rollout ───────────────────────────────────────────────────
import mediapy as media, mujoco, numpy as np

policy_fn = jax.jit(make_policy(params, deterministic=True))
rng = jax.random.PRNGKey(1)
state = env.reset(rng)
renderer = mujoco.Renderer(env.sys.mj_model, height=480, width=640)
mj_data  = mujoco.MjData(env.sys.mj_model)
frames   = []

for _ in range(300):
    rng, key = jax.random.split(rng)
    action, _ = policy_fn(state.obs, key)
    state = env.step(state, action)
    mj_data.qpos[:] = np.array(state.pipeline_state.qpos)
    mj_data.qvel[:] = np.array(state.pipeline_state.qvel)
    mujoco.mj_forward(env.sys.mj_model, mj_data)
    renderer.update_scene(mj_data)
    frames.append(renderer.render())

media.show_video(frames, fps=50)